In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from pyliftover import LiftOver

# Chain file downloaded from:
# https://hgdownload.soe.ucsc.edu/goldenPath/mm10/liftOver/
chain_file = "/gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/liftover/mm10ToMm39.over.chain.gz"
lo = LiftOver(chain_file)

In [3]:
# Load in the RH and DT ATSE files 
# The EasySci ATSEs are in mm9...

WD = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/EasySci2024/LeafletFA/MetaCells/ATSEs"
RH_atses = f"{WD}/RH/EasySci_RH_Metacells_no_annotations_50_500000_10_20241204_single_cell.gz"
DT_atses = f"{WD}/DT/EasySci_DT_Metacells_no_annotations_50_500000_10_20241204_single_cell.gz"

RH_atses = pd.read_csv(RH_atses, sep="}")
DT_atses = pd.read_csv(DT_atses, sep="}")

# Load in ATSEs from Tabula Muris Senis (mm10) 
TMS_atses = ATSE_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/tabula_senis_annotationFREE_intron_clusters_50_500000_100_20250123_single_cell.gz"
TMS_atses = pd.read_csv(TMS_atses, sep="}")

In [4]:
# break up junction_id to get chrom, start, end, strand via "_" split for RH and DT ATSEs
RH_atses[["chrom", "start", "end", "strand"]] = RH_atses.junction_id.str.split("_", expand=True)
RH_atses['chrom'] = RH_atses['chrom'].astype(str)
RH_atses['start'] = RH_atses['start'].astype(int)
RH_atses['end'] = RH_atses['end'].astype(int)

DT_atses[["chrom", "start", "end", "strand"]] = DT_atses.junction_id.str.split("_", expand=True)
DT_atses['chrom'] = DT_atses['chrom'].astype(str)
DT_atses['start'] = DT_atses['start'].astype(int)
DT_atses['end'] = DT_atses['end'].astype(int)

TMS_atses[["chrom", "start", "end", "strand"]] = TMS_atses.junction_id.str.split("_", expand=True)
TMS_atses['chrom'] = TMS_atses['chrom'].astype(str)
TMS_atses['start'] = TMS_atses['start'].astype(int)
TMS_atses['end'] = TMS_atses['end'].astype(int)

In [5]:
# Function to convert a single row of coordinates
def liftover_row(row):
    chrom, start, end = row['chrom'], row['start'], row['end']
    # Lift over start and end coordinates
    new_start = lo.convert_coordinate(chrom, start)
    new_end = lo.convert_coordinate(chrom, end)

    # Check if both start and end were successfully converted
    if new_start and new_end:
        # Extract the new chromosome and positions
        new_chrom = new_start[0][0]
        new_start_pos = int(new_start[0][1])
        new_end_pos = int(new_end[0][1])
        return pd.Series([new_chrom, new_start_pos, new_end_pos])
    else:
        # Return None for unmapped coordinates
        return pd.Series([None, None, None])

# Apply the liftover function to each row
TMS_atses[['new_chrom', 'new_start', 'new_end']] = TMS_atses.apply(liftover_row, axis=1)

# Create the new_junction_id column
TMS_atses['new_junction_id'] = TMS_atses.apply(
    lambda row: (
        f"{row['new_chrom']}_{int(row['new_start'])}_{int(row['new_end'])}_{row['strand']}"
        if pd.notnull(row['new_chrom']) and pd.notnull(row['new_start']) and pd.notnull(row['new_end'])
        else None
    ),
    axis=1
)

In [6]:
# how many junctions are common between RH_atses and DT_atses in junction_id column
common_junctions = set(RH_atses["junction_id"]).intersection(set(DT_atses["junction_id"]))
print(f"Number of common junctions between RH and DT: {len(common_junctions)}")

# how many junctions are unique to RH_atses
unique_junctions_RH = set(RH_atses["junction_id"]).difference(set(DT_atses["junction_id"]))
print(f"Number of unique junctions in RH: {len(unique_junctions_RH)}")

# how many junctions are unique to DT_atses
unique_junctions_DT = set(DT_atses["junction_id"]).difference(set(RH_atses["junction_id"]))
print(f"Number of unique junctions in DT: {len(unique_junctions_DT)}")

Number of common junctions between RH and DT: 8108
Number of unique junctions in RH: 9783
Number of unique junctions in DT: 3566


In [7]:
# How many common junctions are there between TMS_atses and RH_atses
common_junctions = set(TMS_atses["new_junction_id"]).intersection(set(RH_atses["junction_id"]))
print(f"Number of common junctions between TMS and RH: {len(common_junctions)}")

# How many common junctions are there between TMS_atses and DT_atses
common_junctions = set(TMS_atses["new_junction_id"]).intersection(set(DT_atses["junction_id"]))
print(f"Number of common junctions between TMS and DT: {len(common_junctions)}")

# How many common junctions between TMS, RH and DT 
common_junctions = set(TMS_atses["new_junction_id"]).intersection(set(RH_atses["junction_id"])).intersection(set(DT_atses["junction_id"]))
print(f"Number of common junctions between TMS, RH and DT: {len(common_junctions)}")

# How many junctions just in TMS and not in RH or DT
unique_junctions_TMS = set(TMS_atses["new_junction_id"]).difference(set(RH_atses["junction_id"])).difference(set(DT_atses["junction_id"]))
print(f"Number of unique junctions in TMS: {len(unique_junctions_TMS)}")

# Print the number of junctions in each dataset 
print(f"Number of junctions in RH: {len(RH_atses)}")
print(f"Number of junctions in DT: {len(DT_atses)}")
print(f"Number of junctions in TMS: {len(TMS_atses)}")

Number of common junctions between TMS and RH: 10471
Number of common junctions between TMS and DT: 6972
Number of common junctions between TMS, RH and DT: 5434
Number of unique junctions in TMS: 122468
Number of junctions in RH: 17891
Number of junctions in DT: 11674
Number of junctions in TMS: 134645


In [8]:
common_junctions

{'chr9_21930129_21935969_-',
 'chr11_22003555_22006529_-',
 'chr7_101825612_101829583_-',
 'chr19_24075679_24078065_-',
 'chr16_7193833_7209851_+',
 'chr13_101827091_101828690_-',
 'chr15_76982272_76983439_-',
 'chr1_160169734_160178790_-',
 'chr4_43643409_43643605_+',
 'chr11_41803418_41807110_-',
 'chrX_102653022_102656649_-',
 'chr1_80518268_80518944_-',
 'chr19_23191534_23192008_-',
 'chr5_86968545_86969110_+',
 'chr13_74355216_74356776_-',
 'chr13_104281059_104283751_-',
 'chr2_65356130_65356840_-',
 'chr1_151681697_151683116_+',
 'chr4_126088197_126096362_-',
 'chr16_16136705_16137315_-',
 'chr3_130366686_130369023_+',
 'chr8_128125240_128131855_+',
 'chr2_84434096_84435527_-',
 'chr5_44636583_44637567_-',
 'chr10_127961514_127961886_+',
 'chr12_29585367_29616700_+',
 'chr10_90929118_90937616_+',
 'chr18_4379741_4380672_+',
 'chr2_121378504_121379369_-',
 'chrX_104923624_104931320_-',
 'chr2_158218744_158218913_+',
 'chr4_72076654_72087328_-',
 'chr14_30778237_30783425_+',
 'chr7